In [ ]:

# ============================================================
# HYPOTHESIS 2: PARALLEL SIMULATION-BASED POWER ANALYSIS
# HBCD Study Release 2.1
# ============================================================

# If needed, install:
# %pip install numpy pandas scipy statsmodels joblib tqdm
# %pip install threadpoolctl matplotlib

import os

# Set thread limits before importing numerical libraries.
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import norm
from scipy.stats import truncnorm
from statsmodels.regression.mixed_linear_model import MixedLM
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits

# Use text-based tqdm rather than tqdm.auto
from tqdm import tqdm

# ============================================================
# 1. SIMULATION SETTINGS
# ============================================================

N_GM = 1500
N_WM = 1235

N_STATES = 20
N_SITES = 27

# Final analysis: 1000
# Short test: change to 10
N_SIMULATIONS = 1000

EFFECT_SIZES = [
    0.00, 0.05, 0.10, 0.15, 0.20,
    0.25, 0.30, 0.35, 0.40, 0.50
]

ALPHA = 0.05
TARGET_POWER = 0.95
SEED = 12345

N_JOBS = min(4, os.cpu_count() or 1)

# Variance components on standardized outcome scale
VAR_STATE = 0.03
VAR_SITE = 0.04
VAR_FAMILY = 0.03
VAR_RESIDUAL = 0.90

# Assumed fixed effects
BETA_PREDICTOR = 0.10
BETA_PC = 0.05

BETA_GESTATIONAL_AGE = 0.08
BETA_SCAN_AGE = 0.10
BETA_SEX = 0.05
BETA_EDUCATION = 0.05
BETA_ICV = 0.25

# Hypothesis 1 GM assumptions
GM_MEAN = 1220
GM_SD = 187

# Use a distinct directory for each simulation setting.
OUTPUT_DIR = (
    f"HBCD_H2_parallel_power_{N_SIMULATIONS}sim"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

SUMMARY_FILE = os.path.join(
    OUTPUT_DIR, "power_summary.csv"
)

DETAIL_FILE = os.path.join(
    OUTPUT_DIR, "fit_diagnostics.csv"
)

MDE_FILE = os.path.join(
    OUTPUT_DIR, "minimum_detectable_effects.csv"
)

# ============================================================
# 2. ACTUAL STANDARDIZED STATE PC SCORES
# ============================================================

state_data = [
    ("Alabama", 1.551, 0.014),
    ("Arkansas", 1.534, 0.203),
    ("California", -1.175, 1.699),
    ("Georgia", 1.112, 0.445),
    ("Illinois", -0.920, 0.857),
    ("Maryland", -0.970, -1.105),
    ("Massachusetts", -1.238, 0.446),
    ("Minnesota", -0.908, -1.586),
    ("Missouri", 0.624, -0.637),
    ("New Mexico", -0.314, 0.958),
    ("New York", -1.134, 2.054),
    ("North Carolina", 0.672, 0.066),
    ("Ohio", 0.331, -0.033),
    ("Oklahoma", 1.319, 0.149),
    ("Oregon", -1.114, -0.218),
    ("Pennsylvania", 0.339, 0.278),
    ("Tennessee", 1.226, 0.416),
    ("Vermont", -1.199, -1.986),
    ("Virginia", -0.064, -0.591),
    ("Wisconsin", 0.329, -1.429)
]

states_df = pd.DataFrame(
    state_data,
    columns=["state", "PC1", "PC2"]
)

assert len(states_df) == N_STATES

# ============================================================
# 3. DEFINE SIMULATION SCENARIOS
# ============================================================

PREDICTORS = [
    "Income-to-needs ratio",
    "Maternal substance use",
    "Maternal depression"
]

MODERATORS = ["PC1", "PC2"]
OUTCOME_TYPES = ["GM", "WM"]

SCENARIOS = [
    (outcome_type, predictor, moderator, beta)
    for outcome_type in OUTCOME_TYPES
    for predictor in PREDICTORS
    for moderator in MODERATORS
    for beta in EFFECT_SIZES
]

print("=" * 60)
print("HYPOTHESIS 2 POWER ANALYSIS")
print("=" * 60)
print("Number of scenarios:", len(SCENARIOS))
print(
    "Total planned simulations:",
    len(SCENARIOS) * N_SIMULATIONS
)
print("Parallel workers:", N_JOBS)
print("GM sample size:", N_GM)
print("WM sample size:", N_WM)
print("=" * 60)

# ============================================================
# 4. STANDARDIZATION
# ============================================================

def standardize(x):

    x = np.asarray(x, dtype=float)

    sd = np.std(x, ddof=0)

    if not np.isfinite(sd) or sd <= 0:
        raise ValueError("Cannot standardize variable")

    return (x - np.mean(x)) / sd


# ============================================================
# 5. SIMULATE FAMILY CHARACTERISTICS
# ============================================================

def simulate_predictor(n, predictor, rng):

    if predictor == "Income-to-needs ratio":

        # Right-skewed distribution, as in H1.
        # This is an assumed proxy, not empirical HBCD data.
        x = rng.lognormal(
            mean=0,
            sigma=0.75,
            size=n
        )

    elif predictor == "Maternal substance use":

        # Empirical summary:
        # Mean = 4.00
        # SD = 2.88
        # Range = 0 to 18

        mean = 4.00
        variance = 2.88 ** 2

        r = mean ** 2 / (variance - mean)
        p = r / (r + mean)

        x = rng.negative_binomial(
            n=r,
            p=p,
            size=n
        )

        x = np.clip(x, 0, 18)

    elif predictor == "Maternal depression":

        # Empirical summary:
        # Mean = 46.33
        # SD = 8.12
        # Observed range = 38.20 to 81.30
        #
        # Truncated normal approximation.

        mean = 46.33
        sd = 8.12

        lower = (38.20 - mean) / sd
        upper = (81.30 - mean) / sd

        x = truncnorm.rvs(
            lower,
            upper,
            loc=mean,
            scale=sd,
            size=n,
            random_state=rng
        )

    else:
        raise ValueError(
            f"Unknown predictor: {predictor}"
        )

    return standardize(x)


# ============================================================
# 6. SIMULATE MULTILEVEL STRUCTURE
# ============================================================

def simulate_structure(n, rng):

    # Approximately balanced recruitment across states.
    state_idx = np.tile(
        np.arange(N_STATES),
        int(np.ceil(n / N_STATES))
    )[:n]

    rng.shuffle(state_idx)

    # 27 sites distributed across 20 states.
    # This allocation is assumed rather than empirical.
    site_counts = np.array(
        [2] * 7 + [1] * 13
    )

    site_starts = np.cumsum(
        np.r_[0, site_counts[:-1]]
    )

    site_idx = np.empty(n, dtype=int)

    for s in range(N_STATES):

        members = np.where(
            state_idx == s
        )[0]

        site_idx[members] = (
            site_starts[s]
            + rng.integers(
                0,
                site_counts[s],
                size=len(members)
            )
        )

    # Approximately 5% of participants belong
    # to sibling pairs.
    family_idx = np.arange(n)

    n_pairs = int(round(0.05 * n / 2))

    available = {
        s: list(np.where(state_idx == s)[0])
        for s in range(N_STATES)
    }

    for s in available:
        rng.shuffle(available[s])

    for _ in range(n_pairs):

        eligible_states = [
            s for s in range(N_STATES)
            if len(available[s]) >= 2
        ]

        if not eligible_states:
            break

        s = rng.choice(eligible_states)

        a = available[s].pop()
        b = available[s].pop()

        family_idx[b] = family_idx[a]

    df = pd.DataFrame({
        "state_idx": state_idx,
        "site_idx": site_idx,
        "family_idx": family_idx
    })

    # Attach actual state PC scores.
    df["PC1"] = (
        states_df["PC1"].to_numpy()[state_idx]
    )

    df["PC2"] = (
        states_df["PC2"].to_numpy()[state_idx]
    )

    # Simulated covariates
    df["gestational_age"] = standardize(
        rng.normal(39, 1.5, n)
    )

    df["scan_age"] = standardize(
        rng.uniform(0, 12, n)
    )

    df["sex"] = rng.binomial(
        1, 0.5, size=n
    )

    df["maternal_education"] = standardize(
        rng.normal(15, 2.5, n)
    )

    df["ICV"] = standardize(
        rng.normal(0, 1, n)
    )

    # Assumed manufacturer distribution.
    df["scanner_manufacturer"] = rng.choice(
        ["GE", "Siemens", "Philips"],
        size=n,
        p=[0.25, 0.60, 0.15]
    )

    # Placeholder race/ethnicity categories.
    df["race_ethnicity"] = rng.choice(
        ["Group1", "Group2", "Group3", "Group4"],
        size=n,
        p=[0.45, 0.25, 0.20, 0.10]
    )

    return df


# ============================================================
# 7. GENERATE OUTCOME
# ============================================================

def generate_outcome(
    df,
    predictor,
    moderator,
    beta_interaction,
    outcome_type,
    rng
):

    n = len(df)

    x = simulate_predictor(
        n, predictor, rng
    )

    pc = df[moderator].to_numpy()

    state_effects = rng.normal(
        0,
        np.sqrt(VAR_STATE),
        size=N_STATES
    )

    site_effects = rng.normal(
        0,
        np.sqrt(VAR_SITE),
        size=N_SITES
    )

    family_codes, unique_families = pd.factorize(
        df["family_idx"],
        sort=False
    )

    family_effects = rng.normal(
        0,
        np.sqrt(VAR_FAMILY),
        size=len(unique_families)
    )

    residual = rng.normal(
        0,
        np.sqrt(VAR_RESIDUAL),
        size=n
    )

    y = (
        BETA_PREDICTOR * x
        + BETA_PC * pc
        + beta_interaction * x * pc

        + BETA_GESTATIONAL_AGE
        * df["gestational_age"].to_numpy()

        + BETA_SCAN_AGE
        * df["scan_age"].to_numpy()

        + BETA_SEX
        * df["sex"].to_numpy()

        + BETA_EDUCATION
        * df["maternal_education"].to_numpy()

        + state_effects[
            df["state_idx"].to_numpy()
        ]

        + site_effects[
            df["site_idx"].to_numpy()
        ]

        + family_effects[family_codes]

        + residual
    )

    if outcome_type == "GM":

        y += (
            BETA_ICV
            * df["ICV"].to_numpy()
        )

        # Transform standardized-scale volume
        # into the assumed H1 GM measurement scale.
        y = GM_MEAN + GM_SD * y

    result = df.copy()

    result["predictor"] = x
    result["moderator"] = pc
    result["interaction"] = x * pc
    result["outcome"] = y

    return result


# ============================================================
# 8. FIT MIXED-EFFECTS MODEL
# ============================================================

def fit_lme(df, outcome_type):

    formula = (
        "outcome ~ predictor + moderator + interaction"
        " + gestational_age + scan_age + sex"
        " + maternal_education"
        " + C(scanner_manufacturer)"
        " + C(race_ethnicity)"
    )

    if outcome_type == "GM":
        formula += " + ICV"

    model = MixedLM.from_formula(
        formula=formula,
        data=df,
        groups="state_idx",
        re_formula="1",
        vc_formula={
            "site": "0 + C(site_idx)",
            "family": "0 + C(family_idx)"
        }
    )

    for method in ["lbfgs", "powell"]:

        try:

            with warnings.catch_warnings():
                warnings.simplefilter("ignore")

                fit = model.fit(
                    reml=False,
                    method=method,
                    maxiter=500,
                    disp=False
                )

            beta_hat = float(
                fit.params["interaction"]
            )

            se = float(
                fit.bse["interaction"]
            )

            if (
                fit.converged
                and np.isfinite(beta_hat)
                and np.isfinite(se)
                and se > 0
            ):

                z = beta_hat / se
                p_value = 2 * norm.sf(abs(z))

                return {
                    "p_value": p_value,
                    "beta_hat": beta_hat,
                    "se": se,
                    "optimizer": method,
                    "converged": True
                }

        except Exception:
            continue

    return {
        "p_value": np.nan,
        "beta_hat": np.nan,
        "se": np.nan,
        "optimizer": "failed",
        "converged": False
    }


# ============================================================
# 9. RUN ONE SCENARIO
# ============================================================

def run_scenario(scenario, seed):

    outcome_type, predictor, moderator, beta = scenario

    n = N_GM if outcome_type == "GM" else N_WM

    rng = np.random.default_rng(seed)

    details = []
    significant = 0
    valid = 0

    lbfgs_count = 0
    powell_count = 0

    with threadpool_limits(limits=1):

        for sim in range(N_SIMULATIONS):

            structure = simulate_structure(n, rng)

            simulated_df = generate_outcome(
                structure,
                predictor,
                moderator,
                beta,
                outcome_type,
                rng
            )

            fit_result = fit_lme(
                simulated_df,
                outcome_type
            )

            p = fit_result["p_value"]

            if np.isfinite(p):
                valid += 1
                significant += int(p < ALPHA)

            if fit_result["optimizer"] == "lbfgs":
                lbfgs_count += 1

            elif fit_result["optimizer"] == "powell":
                powell_count += 1

            details.append({
                "outcome_type": outcome_type,
                "predictor": predictor,
                "moderator": moderator,
                "beta": beta,
                "simulation": sim + 1,
                **fit_result
            })

    power = (
        significant / valid
        if valid > 0
        else np.nan
    )

    summary = {
        "outcome_type": outcome_type,
        "predictor": predictor,
        "moderator": moderator,
        "beta": beta,
        "power": power,
        "valid_fits": valid,
        "failed_fits": N_SIMULATIONS - valid,
        "lbfgs_fits": lbfgs_count,
        "powell_fits": powell_count,
        "n_simulations": N_SIMULATIONS
    }

    return summary, details


# ============================================================
# 10. PARALLEL EXECUTION WITH VISIBLE PROGRESS
# ============================================================

seed_sequence = np.random.SeedSequence(SEED)

scenario_seeds = seed_sequence.spawn(
    len(SCENARIOS)
)

scenario_jobs = list(
    zip(SCENARIOS, scenario_seeds)
)

# Read completed scenarios.
if os.path.exists(SUMMARY_FILE):

    existing_summary = pd.read_csv(
        SUMMARY_FILE
    )

else:
    existing_summary = pd.DataFrame()

completed = set()

if not existing_summary.empty:

    if not (
        existing_summary["n_simulations"]
        == N_SIMULATIONS
    ).all():

        raise ValueError(
            "Checkpoint simulation counts do not match. "
            "Use a new output directory."
        )

    for _, row in existing_summary.iterrows():

        completed.add((
            row["outcome_type"],
            row["predictor"],
            row["moderator"],
            round(float(row["beta"]), 8)
        ))

pending_jobs = [
    (scenario, seed)
    for scenario, seed in scenario_jobs
    if (
        scenario[0],
        scenario[1],
        scenario[2],
        round(float(scenario[3]), 8)
    ) not in completed
]

print()
print("Completed scenarios:", len(completed))
print("Pending scenarios:", len(pending_jobs))
print("Simulations per scenario:", N_SIMULATIONS)
print()

# Number of parallel batches
n_batches = int(
    np.ceil(len(pending_jobs) / N_JOBS)
)

# Text-based progress bar (no ipywidgets needed)
progress = tqdm(
    total=n_batches,
    desc="Parallel scenario batches",
    ascii=True,
    dynamic_ncols=True,
    leave=True,
    mininterval=1
)

for start in range(
    0,
    len(pending_jobs),
    N_JOBS
):

    batch = pending_jobs[
        start:start + N_JOBS
    ]

    batch_results = Parallel(
        n_jobs=N_JOBS,
        backend="loky",
        verbose=0
    )(
        delayed(run_scenario)(
            scenario,
            seed
        )
        for scenario, seed in batch
    )

    batch_summary = pd.DataFrame([
        result[0]
        for result in batch_results
    ])

    batch_details = pd.DataFrame([
        row
        for result in batch_results
        for row in result[1]
    ])

    # Save summary after each batch.
    batch_summary.to_csv(
        SUMMARY_FILE,
        mode="a",
        header=not os.path.exists(SUMMARY_FILE),
        index=False
    )

    # Save diagnostics after each batch.
    batch_details.to_csv(
        DETAIL_FILE,
        mode="a",
        header=not os.path.exists(DETAIL_FILE),
        index=False
    )

    progress.update(1)

    completed_this_run = min(
        start + len(batch),
        len(pending_jobs)
    )

    total_completed = (
        len(completed) + completed_this_run
    )

    tqdm.write(
        f"Completed {total_completed}/{len(SCENARIOS)} "
        f"scenarios | "
        f"{total_completed * N_SIMULATIONS:,}/"
        f"{len(SCENARIOS) * N_SIMULATIONS:,} "
        f"simulations"
    )

progress.close()

print()
print("=" * 60)
print("PARALLEL SIMULATION COMPLETED")
print("=" * 60)

# ============================================================
# 11. LOAD AND SUMMARIZE RESULTS
# ============================================================

results = pd.read_csv(SUMMARY_FILE)

results = results.sort_values(
    [
        "outcome_type",
        "predictor",
        "moderator",
        "beta"
    ]
).reset_index(drop=True)

print("\nTotal scenarios:", len(results))

print(
    "Total valid fits:",
    int(results["valid_fits"].sum())
)

print(
    "Total failed fits:",
    int(results["failed_fits"].sum())
)

print(
    "L-BFGS fits:",
    int(results["lbfgs_fits"].sum())
)

print(
    "Powell fits:",
    int(results["powell_fits"].sum())
)

display(results.head(10))

# ============================================================
# 12. EMPIRICAL TYPE I ERROR
# ============================================================

null_results = results[
    np.isclose(results["beta"], 0)
].copy()

null_results = null_results.rename(
    columns={"power": "type_I_error"}
)

print("\nEMPIRICAL TYPE I ERROR (beta = 0)")

display(
    null_results[
        [
            "outcome_type",
            "predictor",
            "moderator",
            "type_I_error",
            "valid_fits",
            "failed_fits"
        ]
    ]
)

# ============================================================
# 13. MINIMUM DETECTABLE INTERACTION EFFECTS
# ============================================================

group_cols = [
    "outcome_type",
    "predictor",
    "moderator"
]

detected = (
    results.loc[
        (results["power"] >= TARGET_POWER)
        & (results["beta"] > 0)
    ]
    .groupby(group_cols)["beta"]
    .min()
    .reset_index()
    .rename(columns={
        "beta": "minimum_tested_beta_95_power"
    })
)

all_configs = results[
    group_cols
].drop_duplicates()

detectable = all_configs.merge(
    detected,
    on=group_cols,
    how="left"
)

detectable.to_csv(
    MDE_FILE,
    index=False
)

print("\nMINIMUM TESTED BETA AT 95% POWER")

display(detectable)

# ============================================================
# 14. POWER CURVES
# ============================================================

plt.rcParams.update({
    "font.family": "Helvetica",
    "font.size": 12,
    "pdf.fonttype": 42,
    "savefig.dpi": 1000
})

purple_dark = "#45216F"
purple_light = "#A15BB7"
target_color = "#C64D73"

for outcome_type in OUTCOME_TYPES:

    for predictor in PREDICTORS:

        subset = results[
            (results["outcome_type"] == outcome_type)
            & (results["predictor"] == predictor)
        ]

        fig, ax = plt.subplots(
            figsize=(7, 5)
        )

        for moderator, color, linestyle, marker in [
            ("PC1", purple_dark, "-", "o"),
            ("PC2", purple_light, "--", "s")
        ]:

            d = subset[
                subset["moderator"] == moderator
            ].sort_values("beta")

            ax.plot(
                d["beta"],
                d["power"],
                color=color,
                linestyle=linestyle,
                marker=marker,
                linewidth=2.5,
                markersize=6,
                label=moderator
            )

        ax.axhline(
            TARGET_POWER,
            color=target_color,
            linestyle=":",
            linewidth=2,
            label="95% power"
        )

        ax.set_xlabel(
            "Assumed interaction coefficient (beta)"
        )

        ax.set_ylabel(
            "Estimated statistical power"
        )

        ax.set_title(
            f"{outcome_type}: {predictor}",
            fontsize=14,
            pad=12
        )

        ax.set_xlim(
            min(EFFECT_SIZES),
            max(EFFECT_SIZES)
        )

        ax.set_ylim(0, 1.03)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.legend(
            frameon=False
        )

        plt.tight_layout()

        safe_name = (
            predictor.lower()
            .replace(" ", "_")
            .replace("-", "_")
        )

        base = os.path.join(
            OUTPUT_DIR,
            f"power_{outcome_type}_{safe_name}"
        )

        plt.savefig(
            base + ".png",
            dpi=1000,
            bbox_inches="tight"
        )

        plt.savefig(
            base + ".pdf",
            bbox_inches="tight"
        )

        plt.show()
        plt.close(fig)

print()
print("All results saved to:")
print(os.path.abspath(OUTPUT_DIR))
